# renxt_core
All shared infrastructure: utilities, OAuth, TokenManager, rate-limiter,
fetch engine, schema helpers, Domo write helpers, and the endpoint catalogue.
`%run renxt_core.ipynb` after `%run user_configuration.ipynb`.

In [ ]:
import pandas as pd

# ── File I/O ──────────────────────────────────────────────────────────────────

def _utc_now_ts() -> int:
    return int(time.time())

def _save_json_atomic(path: str, data: dict):
    """Atomic JSON write – prevents corrupt files on interrupted runs."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=".tmp_", suffix=".json",
                               dir=os.path.dirname(path) or ".")
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        os.replace(tmp, path)
    finally:
        try:
            if os.path.exists(tmp):
                os.remove(tmp)
        except Exception:
            pass

def _load_json(path: str, default=None):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return default

def _raise_for_status_with_body(resp: requests.Response, context: str = ""):
    if resp.ok:
        return
    msg = f"{context} HTTP {resp.status_code} for {resp.url}\n"
    try:
        msg += f"Body: {resp.json()}"
    except Exception:
        msg += f"Body: {resp.text[:800]}"
    raise requests.HTTPError(msg, response=resp)

print("✅ utilities loaded")


In [ ]:
# ── Refresh token persistence via Domo dataset ────────────────────────────────
# domojupyter has no write-back to the account store (get_account_property_value
# is read-only). Instead we persist the refresh token in a single-row Domo
# dataset called TOKEN_STORE_DATASET.
#
# First run: the dataset is created automatically on first write.
# Every run: we read the token at startup, refresh it, and write the new one back.

TOKEN_STORE_DATASET = "renxt_token_store"   # change name if preferred

def _load_persisted_refresh_token() -> str:
    """Read the refresh token from the Domo token store dataset."""
    try:
        df = domo.read_dataframe(TOKEN_STORE_DATASET, query="SELECT * FROM table")
        if not df.empty and "refresh_token" in df.columns:
            val = df["refresh_token"].iloc[0]
            if val and str(val).strip() not in ("", "nan", "None"):
                return str(val).strip()
    except Exception:
        pass
    return ""

def _persist_refresh_token(new_refresh_token: str):
    """Write the latest refresh token back to the Domo token store dataset.
    Called automatically after every successful token refresh so the next
    scheduled run can pick it up even after the container is destroyed.
    """
    try:
        import pandas as _pd
        df = _pd.DataFrame([{"refresh_token": new_refresh_token}])
        domo.write_dataframe(df, dataset=TOKEN_STORE_DATASET, update_method="REPLACE")
    except Exception as e:
        # Non-fatal — the current run continues, but the NEXT run may fail
        # to obtain a token if the refresh token rotated and was not saved.
        print(f"⚠️  Could not persist refresh token to {TOKEN_STORE_DATASET}: {e}")

print("✅ token persistence helpers loaded")
print(f"   Token store dataset : {TOKEN_STORE_DATASET}")


In [ ]:
# ── OAuth helpers ─────────────────────────────────────────────────────────────

def _generate_pkce():
    verifier  = secrets.token_urlsafe(64)[:128]
    challenge = base64.urlsafe_b64encode(
        hashlib.sha256(verifier.encode()).digest()
    ).decode().rstrip("=")
    return verifier, challenge

def _normalize_tokens(tokens: dict, prior: Optional[dict] = None) -> dict:
    prior = prior or {}
    t = dict(tokens or {})
    if "obtained_at" not in t:
        t["obtained_at"] = _utc_now_ts()
    if not t.get("refresh_token") and prior.get("refresh_token"):
        t["refresh_token"] = prior["refresh_token"]
    return t

def _token_expiring_soon(tokens: dict, skew: int = 120) -> bool:
    if not tokens or not tokens.get("access_token"):
        return True
    expires_in  = tokens.get("expires_in")
    obtained_at = tokens.get("obtained_at")
    if not expires_in or not obtained_at:
        return False
    return _utc_now_ts() >= int(obtained_at) + int(expires_in) - skew

def _is_token_expired_response(resp: requests.Response) -> bool:
    return resp.status_code in (401, 403)

def refresh_tokens(refresh_token: str,
                   preserve: bool = True,
                   prior_tokens: Optional[dict] = None) -> dict:
    """Exchange a refresh token for a new access token and persist the result."""
    if not CLIENT_ID or not CLIENT_SECRET:
        raise RuntimeError("CLIENT_ID and CLIENT_SECRET must be set.")
    if not refresh_token:
        raise RuntimeError("No refresh_token provided.")

    data = {"grant_type": "refresh_token", "refresh_token": refresh_token.strip()}
    if preserve:
        data["preserve_refresh_token"] = "true"

    resp = requests.post(
        TOKEN_URL, data=data,
        auth=(CLIENT_ID.strip(), CLIENT_SECRET.strip()),
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        timeout=60,
    )
    if not resp.ok:
        try:    err = resp.json()
        except: err = resp.text
        raise requests.HTTPError(
            f"Token refresh failed: HTTP {resp.status_code} – {err}", response=resp)

    tokens = _normalize_tokens(resp.json(),
                               prior=prior_tokens or {"refresh_token": refresh_token})

    # Persist so the next scheduled run can use it
    if tokens.get("refresh_token"):
        _persist_refresh_token(tokens["refresh_token"])

    _save_json_atomic(TOKEN_PATH, tokens)
    return tokens

def authorize_interactive():
    """INTERACTIVE ONLY – run once manually to obtain the first refresh token."""
    verifier, challenge = _generate_pkce()
    params = {"response_type":"code","client_id":CLIENT_ID,"redirect_uri":REDIRECT_URI,
              "code_challenge":challenge,"code_challenge_method":"S256"}
    print("\n=== INTERACTIVE AUTH ===")
    print("1) Open this URL and complete consent:")
    print(f"   {AUTH_URL}?{urlencode(params)}")
    print("2) Paste the FULL redirected URL below.\n")

    from urllib.parse import urlparse, parse_qs
    redirected = input("Redirected URL: ").strip()
    q    = parse_qs(urlparse(redirected).query)
    code = q.get("code", [None])[0]
    if not code:
        raise RuntimeError("No authorization code found in URL.")

    resp = requests.post(
        TOKEN_URL,
        data={"grant_type":"authorization_code","client_id":CLIENT_ID,
              "redirect_uri":REDIRECT_URI,"code_verifier":verifier,"code":code},
        auth=(CLIENT_ID.strip(), CLIENT_SECRET.strip()),
        headers={"Content-Type":"application/x-www-form-urlencoded"},
        timeout=60,
    )
    resp.raise_for_status()
    tokens = _normalize_tokens(resp.json())
    _save_json_atomic(TOKEN_PATH, tokens)
    if tokens.get("refresh_token"):
        _persist_refresh_token(tokens["refresh_token"])
        print("✅ Tokens obtained. refresh_token saved to Domo account store.")
    return tokens

def ensure_tokens(*, interactive: bool = False) -> dict:
    """Return valid tokens, refreshing automatically. For scheduled runs: interactive=False."""
    # Token priority: env/account-store preset → dataset store → local file cache
    rt = (PRESET_REFRESH_TOKEN
          or _load_persisted_refresh_token()
          or (_load_json(TOKEN_PATH, {}) or {}).get("refresh_token"))
    if rt:
        return refresh_tokens(rt, preserve=PRESERVE_REFRESH_TOKEN)
    cached = _load_json(TOKEN_PATH, {}) or {}
    if cached.get("access_token") and not _token_expiring_soon(cached):
        return cached
    if interactive:
        return authorize_interactive()
    raise RuntimeError(
        "Headless run: no refresh_token found.\n"
        "Run first_time_auth.ipynb once to bootstrap tokens."
    )

print("✅ OAuth helpers loaded")


In [ ]:
# ── TokenManager ──────────────────────────────────────────────────────────────

class TokenManager:
    """
    Manages access/refresh tokens for scheduled and interactive runs.
    - Bootstraps via ensure_tokens() at construction.
    - Pre-emptively refreshes before expiry.
    - Auto-refreshes on 401/403.
    - All refreshes write back to the Domo account store.
    """

    def __init__(self, interactive: bool = False):
        self.interactive   = interactive
        self.tokens        = ensure_tokens(interactive=interactive)
        self.tokens        = _normalize_tokens(self.tokens, prior=self.tokens)
        self.access_token  = self.tokens["access_token"]
        self.refresh_token = self.tokens.get("refresh_token", "")
        if not self.access_token:
            raise RuntimeError("No access_token available.")

    def refresh(self, reason: str = "", force: bool = False):
        if not force and not _token_expiring_soon(self.tokens):
            return
        rt = self.refresh_token or PRESET_REFRESH_TOKEN
        if not rt:
            if not self.interactive:
                raise RuntimeError(
                    f"Headless refresh failed – no refresh_token. Reason: {reason}. "
                    "Run first_time_auth.ipynb to re-bootstrap.")
            self.tokens        = authorize_interactive()
            self.tokens        = _normalize_tokens(self.tokens, prior=self.tokens)
            self.access_token  = self.tokens["access_token"]
            self.refresh_token = self.tokens.get("refresh_token", "")
            return
        self.tokens        = refresh_tokens(rt, preserve=PRESERVE_REFRESH_TOKEN,
                                            prior_tokens=self.tokens)
        self.tokens        = _normalize_tokens(self.tokens, prior=self.tokens)
        self.access_token  = self.tokens["access_token"]
        self.refresh_token = self.tokens.get("refresh_token", rt)

print("✅ TokenManager loaded")


In [ ]:
# ── Rate limiter & HTTP helpers ───────────────────────────────────────────────

_last_sec       = 0
_calls_this_sec = 0

def _rate_limit_guard():
    global _last_sec, _calls_this_sec
    now = time.time()
    if int(now) != int(_last_sec):
        _last_sec = now
        _calls_this_sec = 0
    if _calls_this_sec >= CALLS_PER_SECOND:
        sleep_for = 1.0 - (now - int(now))
        if sleep_for > 0:
            time.sleep(sleep_for)
        _last_sec = time.time()
        _calls_this_sec = 0
    _calls_this_sec += 1

def api_request(method, url, headers=None, params=None,
                json_body=None, session=None) -> requests.Response:
    """Core request: throttles, retries on 429/5xx."""
    sess    = session or requests.Session()
    hdrs    = {"Bb-Api-Subscription-Key": SUBSCRIPTION_KEY, "Accept": "application/json"}
    if headers:
        hdrs.update(headers)
    backoff  = INITIAL_BACKOFF_SECONDS
    last_exc = None
    for attempt in range(1, MAX_RETRIES + 1):
        _rate_limit_guard()
        try:
            resp = sess.request(method, url, headers=hdrs,
                                params=params, json=json_body, timeout=60)
        except requests.RequestException as exc:
            last_exc = exc
            time.sleep(backoff)
            backoff = min(backoff * 2, 60)
            continue
        if resp.status_code == 429:
            time.sleep(float(resp.headers.get("Retry-After", backoff)))
            backoff = min(backoff * 2, 60)
            continue
        if 500 <= resp.status_code < 600:
            time.sleep(backoff)
            backoff = min(backoff * 2, 60)
            continue
        return resp
    if last_exc:
        raise last_exc
    return resp

def api_request_with_auth(method, url, token_mgr: TokenManager,
                          headers=None, params=None,
                          json_body=None, session=None) -> requests.Response:
    """Adds Bearer auth; auto-refreshes on 401/403 and retries once."""
    sess = session or requests.Session()
    token_mgr.refresh(reason="preflight", force=False)
    hdrs = dict(headers or {})
    hdrs["Authorization"] = f"Bearer {token_mgr.access_token}"
    resp = api_request(method, url, headers=hdrs, params=params,
                       json_body=json_body, session=sess)
    if _is_token_expired_response(resp):
        token_mgr.refresh(reason=f"HTTP {resp.status_code}", force=True)
        hdrs["Authorization"] = f"Bearer {token_mgr.access_token}"
        resp = api_request(method, url, headers=hdrs, params=params,
                           json_body=json_body, session=sess)
    return resp

print("✅ rate-limiter + HTTP helpers loaded")


In [ ]:
# ── Endpoint catalogue ────────────────────────────────────────────────────────
# Canonical endpoint names used throughout this codebase.
# NAMING RULES:
#   - Names match the last segment of the Blackbaud API path (no underscores
#     unless the path itself uses them).
#   - "emailaddresses" (no underscore) = /constituent/v1/emailaddresses
#   - "opportunity"   (singular)       = /opportunity/v1/opportunities
#   - "gift_customfields"              = /gift/v1/gifts/customfields

ENDPOINTS = [
    {"name": "constituents",
     "path": "/constituent/v1/constituents",
     "params_base": {"include_inactive": "true", "include_deceased": "true"},
     "incremental_candidates": ["last_modified", "date_added"]},

    {"name": "actions",
     "path": "/constituent/v1/actions",
     "params_base": {},
     "incremental_candidates": ["last_modified", "date_added"]},

    {"name": "addresses",
     "path": "/constituent/v1/addresses",
     "params_base": {"include_inactive": "true"},
     "incremental_candidates": ["last_modified", "date_added"]},

    {"name": "constituentcodes",
     "path": "/constituent/v1/constituents/constituentcodes",
     "params_base": {"include_inactive": "true"},
     "incremental_candidates": ["last_modified", "date_added"]},

    {"name": "customfields",
     "path": "/constituent/v1/constituents/customfields",
     "params_base": {},
     "incremental_candidates": ["last_modified", "date_added"]},

    {"name": "educations",
     "path": "/constituent/v1/educations",
     "params_base": {},
     "incremental_candidates": ["last_modified", "date_added"]},

    # "emailaddresses" — no underscore, matches /constituent/v1/emailaddresses
    {"name": "emailaddresses",
     "path": "/constituent/v1/emailaddresses",
     "params_base": {"include_inactive": "true"},
     "incremental_candidates": ["last_modified", "date_added"]},

    {"name": "memberships",
     "path": "/constituent/v1/memberships",
     "params_base": {},
     "incremental_candidates": ["last_modified", "date_added"]},

    {"name": "notes",
     "path": "/constituent/v1/notes",
     "params_base": {},
     "incremental_candidates": ["last_modified", "date_added"]},

    {"name": "onlinepresences",
     "path": "/constituent/v1/onlinepresences",
     "params_base": {"include_inactive": "true"},
     "incremental_candidates": ["last_modified", "date_added"]},

    {"name": "phones",
     "path": "/constituent/v1/phones",
     "params_base": {"include_inactive": "true"},
     "incremental_candidates": ["last_modified", "date_added"]},

    {"name": "relationships",
     "path": "/constituent/v1/relationships",
     "params_base": {},
     "incremental_candidates": ["last_modified", "date_added"]},

    # "opportunity" — singular key, maps to /opportunity/v1/opportunities
    {"name": "opportunity",
     "path": "/opportunity/v1/opportunities",
     "params_base": {"include_inactive": "true"},
     "incremental_candidates": ["last_modified", "date_added"]},

    {"name": "gifts",
     "path": "/gift/v1/gifts",
     "params_base": {"include_inactive": "true", "limit": str(LIMIT)},
     "incremental_candidates": ["last_modified", "date_added"]},

    {"name": "gift_customfields",
     "path": "/gift/v1/gifts/customfields",
     "params_base": {},
     "incremental_candidates": ["last_modified", "date_added"]},

    {"name": "appeal",
     "path": "/fundraising/v1/appeals",
     "params_base": {},
     "incremental_candidates": []},

    {"name": "campaign",
     "path": "/fundraising/v1/campaigns",
     "params_base": {},
     "incremental_candidates": []},

    {"name": "fund",
     "path": "/fundraising/v1/funds",
     "params_base": {},
     "incremental_candidates": []},
]

ENDPOINT_LOOKUP = {e["name"]: e for e in ENDPOINTS}
print(f"✅ {len(ENDPOINTS)} endpoints registered: {list(ENDPOINT_LOOKUP)}")


In [ ]:
# ── Schema + DataFrame helpers ────────────────────────────────────────────────

def load_schemas(path: str = "schemas.json") -> dict:
    return _load_json(path, default={}) or {}

def apply_schema(df: pd.DataFrame, endpoint_name: str, schemas: dict) -> pd.DataFrame:
    """Cast columns and enforce ordering from schemas.json."""
    spec      = (schemas or {}).get(endpoint_name) or {}
    cols_spec = spec.get("columns") or {}
    order     = spec.get("order") or []
    out = df.copy()
    for col, meta in cols_spec.items():
        if col not in out.columns:
            continue
        t = str((meta or {}).get("type", "")).lower()
        if   t == "datetime": out[col] = pd.to_datetime(out[col], errors="coerce", utc=True)
        elif t == "number":   out[col] = pd.to_numeric(out[col], errors="coerce")
        elif t == "boolean":  out[col] = out[col].astype("boolean")
        elif t == "string":   out[col] = out[col].astype("string")
    if order:
        existing  = [c for c in order if c in out.columns]
        remaining = [c for c in out.columns if c not in existing]
        out = out[existing + remaining]
    return out

def domo_safe_cast(df: pd.DataFrame,
                   force_str_cols=("id", "constituent_id", "lookup_id"),
                   drop_tz: bool = True) -> pd.DataFrame:
    """Prepare a DataFrame for domo.write_dataframe():
    - Drop timezone info from datetime columns (Domo does not support tz-aware).
    - Convert Pandas StringDtype / BooleanDtype to plain object.
    - Force ID columns to object strings.
    """
    out = df.copy()
    for col in out.columns:
        dtype_str = str(out[col].dtype)
        if drop_tz and hasattr(out[col], "dt") and getattr(out[col].dt, "tz", None):
            out[col] = out[col].dt.tz_convert("UTC").dt.tz_localize(None)
        if dtype_str in ("string", "StringDtype"):
            out[col] = out[col].where(out[col].isna(), out[col].astype(str)).astype(object)
        if dtype_str in ("boolean", "bool"):
            out[col] = out[col].astype(object)
    for col in force_str_cols:
        if col in out.columns:
            out[col] = out[col].map(lambda x: x if pd.isna(x) else str(x)).astype(object)
    return out

def ensure_columns(df: pd.DataFrame, columns: list, fill_value=pd.NA) -> pd.DataFrame:
    """Guarantee columns exist and reorder to match `columns`."""
    df = df.copy()
    for c in columns:
        if c not in df.columns:
            df[c] = fill_value
    return df.reindex(columns=columns)

print("✅ schema + DataFrame helpers loaded")


In [ ]:
# ── Fetch engine ──────────────────────────────────────────────────────────────

try:
    from tqdm import tqdm as _tqdm
    _HAS_TQDM = True
except ImportError:
    _HAS_TQDM = False

def compute_since_utc(days_back: int) -> str:
    """Return MM-DD-YYYY string `days_back` days ago (UTC)."""
    return (datetime.now(timezone.utc) - timedelta(days=int(days_back))).strftime("%m-%d-%Y")

def extract_items_and_next(payload: dict) -> tuple:
    """Handle both SKY list envelopes: {value:[...]} and {items:[...]}."""
    if not isinstance(payload, dict):
        return [], None
    items = None
    for k in ("value", "items", "results"):
        if isinstance(payload.get(k), list):
            items = payload[k]
            break
    return (items or []), payload.get("next_link") or payload.get("next")

def fetch_incremental(token_mgr: TokenManager,
                      endpoint_name: str,
                      days_back: int,
                      incremental_field: Optional[str] = None,
                      session=None) -> pd.DataFrame:
    """
    Pull a list endpoint from Blackbaud SKY API.
    days_back=0  → full historic pull (no date filter)
    days_back>0  → incremental pull filtered by last_modified >= (now - days_back)
    """
    if endpoint_name not in ENDPOINT_LOOKUP:
        raise ValueError(f"Unknown endpoint '{endpoint_name}'. "
                         f"Available: {list(ENDPOINT_LOOKUP)}")

    cfg       = ENDPOINT_LOOKUP[endpoint_name]
    cands     = cfg.get("incremental_candidates") or []
    inc_field = incremental_field or (cands[0] if cands else None)
    params    = dict(cfg.get("params_base") or {})

    if days_back > 0 and inc_field:
        params[inc_field] = compute_since_utc(days_back)
        filter_info = f"{inc_field}={params[inc_field]}"
    else:
        filter_info = "historic pull (no date filter)"

    url      = f"{API_BASE}{cfg['path']}"
    sess     = session or requests.Session()
    all_rows = []
    page     = 0
    pbar     = None

    print(f"🔄 Fetching [{endpoint_name}] | {filter_info}")

    while True:
        page += 1
        resp = api_request_with_auth("GET", url, token_mgr=token_mgr,
                                     params=params, session=sess)
        _raise_for_status_with_body(resp, context=f"[{endpoint_name}] page={page}")

        payload          = resp.json() if resp.text else {}
        items, next_link = extract_items_and_next(payload)
        all_rows.extend(items)

        if pbar is None and _HAS_TQDM:
            total = payload.get("count")
            if total:
                pbar = _tqdm(total=int(total), desc=endpoint_name,
                             unit="rec", mininterval=30, leave=False)
        if pbar:
            pbar.update(len(items))

        params = None  # next_link encodes params — don't double-send
        if not items or not next_link:
            break
        url = next_link if next_link.startswith("http") else urljoin(API_BASE, next_link)

    if pbar:
        pbar.close()

    if not all_rows:
        print(f"  ⚠️  No records returned for [{endpoint_name}]")
        return pd.DataFrame()

    df = pd.json_normalize(all_rows, sep=".")
    df["_endpoint"]          = endpoint_name
    df["_days_back"]         = days_back
    df["_incremental_field"] = inc_field or ""
    df["pulled_at_utc"]      = datetime.now(timezone.utc).isoformat()
    print(f"  ✅ [{endpoint_name}] {len(df):,} records")
    return df

print("✅ fetch engine loaded")


In [ ]:
# ── Smart Domo upsert ─────────────────────────────────────────────────────────
# Domo's write_dataframe does not do a true server-side upsert.
# We implement it by: read existing → merge on key → write full result back.
# This preserves non-null values in columns that incremental pulls may omit.

def smart_upsert_domo(new_df: pd.DataFrame,
                      dataset_id: str,
                      merge_key: str = "id") -> pd.DataFrame:
    """Read existing data, merge with new_df on merge_key, write full result back.

    Logic:
      - Rows whose ID exists in both → new values win, existing fills any nulls
      - Rows whose ID is only in new  → inserted
      - Rows whose ID is only in existing → kept unchanged
    """
    new_df = domo_safe_cast(new_df.copy())

    try:
        existing_df = domo.read_dataframe(dataset_id, query="SELECT * FROM table")
        print(f"  📥 Existing rows: {len(existing_df):,}")
    except Exception as e:
        print(f"  ℹ️  No existing data ({e}); treating as first load.")
        existing_df = pd.DataFrame()

    if existing_df.empty:
        domo.write_dataframe(new_df, dataset=dataset_id, update_method="REPLACE")
        print(f"  ✅ First load: wrote {len(new_df):,} rows → {dataset_id}")
        return new_df

    existing_df[merge_key] = existing_df[merge_key].astype(str)
    new_df[merge_key]      = new_df[merge_key].astype(str)

    existing_ids = set(existing_df[merge_key])
    new_ids      = set(new_df[merge_key])

    # Rows in existing that are NOT in the new pull — keep them untouched
    to_keep = existing_df[~existing_df[merge_key].isin(new_ids)].copy()

    # Rows in existing that ARE in the new pull — merge: new wins, existing fills nulls
    to_update = existing_df[existing_df[merge_key].isin(new_ids)].copy()

    # Rows in new pull — split into updates vs brand-new inserts
    new_updates = new_df[new_df[merge_key].isin(existing_ids)].copy()
    truly_new   = new_df[~new_df[merge_key].isin(existing_ids)].copy()

    print(f"  🔄 {len(new_updates):,} rows updated  "
          f"➕ {len(truly_new):,} inserted  🔒 {len(to_keep):,} unchanged")

    if not to_update.empty:
        # new_updates and to_update have the same IDs — merge on index
        merged_updates = (
            new_updates
            .set_index(merge_key)
            .combine_first(to_update.set_index(merge_key))
            .reset_index()
        )
    else:
        # Nothing to update — all new_df rows are brand-new
        merged_updates = new_df.copy()
        truly_new      = pd.DataFrame(columns=new_df.columns)

    # Final dataset: updated rows + unchanged existing rows + brand-new rows
    final_df = pd.concat([merged_updates, to_keep, truly_new], ignore_index=True)
    final_df = domo_safe_cast(final_df)
    domo.write_dataframe(final_df, dataset=dataset_id, update_method="REPLACE")
    print(f"  ✅ Wrote {len(final_df):,} total rows → {dataset_id}")
    return final_df

print("✅ smart_upsert_domo loaded")


In [ ]:
# ── Watermark / pipeline state ────────────────────────────────────────────────
# renxt_pipeline_state is a Domo dataset with one row per endpoint.
# Schema (create manually in Domo, register as Input + Output on every workspace):
#   endpoint               (text)
#   last_successful_run_utc (text)  — date string YYYY-MM-DD
#   last_run_status        (text)  — "success" | "failed"
#   rows_written           (decimal)
#   updated_at             (text)  — full ISO timestamp for visibility

PIPELINE_STATE_DATASET = "renxt_pipeline_state"
WATERMARK_FALLBACK_DAYS = 7   # used when no prior successful run exists


def _read_pipeline_state() -> pd.DataFrame:
    """Read the full state table. Returns empty DataFrame if not found."""
    try:
        df = domo.read_dataframe(PIPELINE_STATE_DATASET, query="SELECT * FROM table")
        return df
    except Exception:
        return pd.DataFrame(columns=[
            "endpoint", "last_successful_run_utc",
            "last_run_status", "rows_written", "updated_at"
        ])


def _get_since_date(endpoint_name: str, state_df: pd.DataFrame) -> str:
    """
    Return the MM-DD-YYYY date to use as the incremental filter.
    Uses last_successful_run_utc from state, falls back to WATERMARK_FALLBACK_DAYS.
    """
    if not state_df.empty and "endpoint" in state_df.columns:
        row = state_df[state_df["endpoint"] == endpoint_name]
        if not row.empty:
            val = row["last_successful_run_utc"].iloc[0]
            if val and str(val).strip() not in ("", "nan", "None"):
                try:
                    # Parse stored YYYY-MM-DD and convert to MM-DD-YYYY for Blackbaud
                    from datetime import datetime as _dt
                    dt = _dt.strptime(str(val).strip()[:10], "%Y-%m-%d")
                    result = dt.strftime("%m-%d-%Y")
                    print(f"  📅 Watermark found: {val} → using since={result}")
                    return result
                except Exception:
                    pass

    # No watermark — fall back
    fallback = compute_since_utc(WATERMARK_FALLBACK_DAYS)
    print(f"  📅 No watermark for [{endpoint_name}] — using {WATERMARK_FALLBACK_DAYS}d fallback: {fallback}")
    return fallback


def _write_pipeline_state(endpoint_name: str, status: str,
                           rows_written: int, state_df: pd.DataFrame) -> None:
    """
    Upsert one row into renxt_pipeline_state.
    On success: updates last_successful_run_utc to today.
    On failure: updates last_run_status but does NOT advance the watermark.
    """
    now_ts  = datetime.now(timezone.utc)
    today   = now_ts.strftime("%Y-%m-%d")
    updated = now_ts.isoformat()

    # Build the new/updated row
    new_row = pd.DataFrame([{
        "endpoint":                endpoint_name,
        "last_successful_run_utc": today if status == "success" else (
            # preserve existing watermark on failure
            state_df.loc[state_df["endpoint"] == endpoint_name,
                         "last_successful_run_utc"].iloc[0]
            if not state_df.empty
            and "endpoint" in state_df.columns
            and not state_df[state_df["endpoint"] == endpoint_name].empty
            else ""
        ),
        "last_run_status": status,
        "rows_written":    rows_written,
        "updated_at":      updated,
    }])

    # Merge with existing state (upsert on endpoint name)
    if state_df.empty or "endpoint" not in state_df.columns:
        final_state = new_row
    else:
        other_rows = state_df[state_df["endpoint"] != endpoint_name].copy()
        final_state = pd.concat([other_rows, new_row], ignore_index=True)

    try:
        domo.write_dataframe(
            domo_safe_cast(final_state),
            dataset=PIPELINE_STATE_DATASET,
            update_method="REPLACE",
        )
        print(f"  📝 Pipeline state updated [{endpoint_name}] → {status}")
    except Exception as e:
        print(f"  ⚠️  Could not update pipeline state: {e}")


print("✅ watermark helpers loaded")
print(f"   State dataset       : {PIPELINE_STATE_DATASET}")
print(f"   Fallback days       : {WATERMARK_FALLBACK_DAYS}")


In [ ]:
# ── run_incremental: full pipeline in one call ────────────────────────────────

def run_incremental(endpoint_name: str,
                    domo_dataset_id: str,
                    update_method: str = "smart_upsert",
                    merge_key: str = "id",
                    incremental_field: Optional[str] = None) -> pd.DataFrame:
    """
    Complete pipeline:
      1. Read pipeline state to get watermark (last successful run date)
      2. Bootstrap tokens
      3. Fetch from Blackbaud SKY API using watermark as since filter
      4. Apply schemas.json transformations
      5. Write to Domo:
           "smart_upsert" → read-merge-replace (preserves non-null history)
           "upsert"       → native Domo upsert on merge_key (faster, no read)
           "REPLACE"      → full overwrite (for reference endpoints with no incremental field)
      6. Update pipeline state:
           - success → advances watermark to today
           - failure → preserves existing watermark (re-fetches from last good point next run)

    Note: days_back parameter removed — date range is driven by watermark.
    REPLACE endpoints (appeal, campaign, fund) always fetch everything
    regardless of watermark since they have no incremental field.
    """
    # ── Step 1: read watermark ──────────────────────────────────────────────
    state_df = _read_pipeline_state()

    # ── Step 2: determine since date ───────────────────────────────────────
    cfg       = ENDPOINT_LOOKUP.get(endpoint_name, {})
    has_incr  = bool(cfg.get("incremental_candidates"))

    if has_incr:
        since_date    = _get_since_date(endpoint_name, state_df)
        inc_field     = incremental_field or (cfg["incremental_candidates"][0])
        fetch_params  = {"days_back": 0}   # we pass since directly via incremental_field override
    else:
        # No incremental field — always full pull, watermark not used for filtering
        since_date   = None
        inc_field    = None
        fetch_params = {"days_back": 0}

    # ── Step 3: fetch ───────────────────────────────────────────────────────
    token_mgr = TokenManager(interactive=False)
    sess      = requests.Session()

    if has_incr and since_date:
        # Inject the since date directly into params_base for this fetch
        cfg_override = dict(cfg)
        cfg_override["params_base"] = dict(cfg.get("params_base") or {})
        cfg_override["params_base"][inc_field] = since_date
        # Temporarily override ENDPOINT_LOOKUP for this call
        ENDPOINT_LOOKUP[endpoint_name] = cfg_override
        try:
            df = fetch_incremental(
                token_mgr=token_mgr,
                endpoint_name=endpoint_name,
                days_back=0,   # date already injected above
                incremental_field=None,
                session=sess,
            )
        finally:
            # Restore original config
            ENDPOINT_LOOKUP[endpoint_name] = cfg
    else:
        df = fetch_incremental(
            token_mgr=token_mgr,
            endpoint_name=endpoint_name,
            days_back=0,
            incremental_field=None,
            session=sess,
        )

    if df.empty:
        print(f"  ⚠️  No data fetched for [{endpoint_name}]")
        # Still update state so we know the run happened (but rows=0)
        _write_pipeline_state(endpoint_name, "success", 0, state_df)
        return df

    # ── Step 4: apply schema ────────────────────────────────────────────────
    df = apply_schema(df, endpoint_name, load_schemas())

    # ── Step 5: write to Domo ───────────────────────────────────────────────
    try:
        if update_method == "smart_upsert":
            result_df = smart_upsert_domo(df, dataset_id=domo_dataset_id,
                                          merge_key=merge_key)
        elif update_method == "upsert":
            result_df = domo_safe_cast(df)
            domo.write_dataframe(result_df, dataset=domo_dataset_id,
                                 update_method="upsert", update_key=merge_key)
            print(f"  ✅ Upserted {len(result_df):,} rows → {domo_dataset_id} [key={merge_key}]")
        else:
            # REPLACE
            result_df = domo_safe_cast(df)
            domo.write_dataframe(result_df, dataset=domo_dataset_id,
                                 update_method="REPLACE")
            print(f"  ✅ Wrote {len(result_df):,} rows → {domo_dataset_id} [REPLACE]")

    except Exception as e:
        # Write failed — record failure but do NOT advance watermark
        print(f"  ❌ Domo write failed: {e}")
        _write_pipeline_state(endpoint_name, "failed", 0, state_df)
        raise

    # ── Step 6: advance watermark ───────────────────────────────────────────
    _write_pipeline_state(endpoint_name, "success", len(result_df), state_df)
    return result_df


print("✅ run_incremental loaded")
print()
print("━" * 60)
print("renxt_core ready.")
print("━" * 60)
